In [2]:
import pandas as pd
import numpy as np
import json
import os
import re

In [3]:
amino_acid_sequence_df = pd.read_csv('Amino_acid_sequence.csv')
amino_acid_sequence_df.rename(columns={'Protein ID':'Protein_id','Sequence':'Amino_acid_sequence'}, inplace=True)

amino_acid_sequence_df['Protein_id'] = amino_acid_sequence_df['Protein_id'].str[:-2]
amino_acid_sequence_df

,Protein_id,Amino_acid_sequence
0,XP_001727958,MSPHEVIGTVPKNSTTFRTQADEHDDHEEALQNLRTGKYEDWPNEA...
1,XP_001727959,MTTPEQDGAARLAARKPIPKPVPAYLPTAGSPLSVDKNLYSTIQQA...
2,XP_001727960,MKFGKQIQKRQLEVPEYAASFVNYKALKKLIKKLSATPILPPQTDL...
3,XP_001727961,MSAPTDSNVSKHATVEPAPESPTTARPFELDDDEAQENVPLGTDTT...
4,XP_001727966,MKLEQSSTPKITGLIRILTLTSEEENRAQSDQVQAIIRDRLRNLNY...
...,...,...
10807,YP_009126725,MPQLVPFYFVNEITFTFVIITLMVYILSKYILPRFVRLFLSRTFIS...
10808,YP_009126726,MSTLSFNNISTEVLSPLNQFEIRDLLSIDALGNLHISITNIGFYLT...
10809,YP_009126727,MSTLSFNNISTEVLSPLNQFEIRDLLSIDALGNLHISITNIGFYLT...
10810,YP_009126728,MIQVAKIIGTGLATTGLIGAGIGIGVVFGSLIIGVSRNPSLKSQLF...


In [4]:
nucleic_acid_df = pd.read_csv('Nucleic_acid_sequence.csv')
nucleic_acid_df.rename(columns={'gene':'Gene','sequence':'Nucleic_acid_sequence'}, inplace=True)

nucleic_acid_df

,Gene,Nucleic_acid_sequence
0,NCU10129,ATGCCTCTCTTCTCCCGCCACGCCGAGCCTGAGCCCGCCCCGGCCC...
1,NCU09901,ATGGCCTCCCGCTCGAGGTCTCGGTCGGCCGACCGCTACGATGACG...
2,NCU09903,ATGACAGTCCCTATCATGGTCGCTGATGGCGCCGCGCCGCCCGTCA...
3,NCU11134,ATGGTTGTTCAAGCTCCCATGCTACAAGAGCCCTCACTGGTTCCAC...
4,NCU09904,ATGGACCACAGACACACCATGGACAACGACCACCGCGAACATCTCG...
...,...,...
10807,NCU16024,ATGCCTCAATTAGTTCCATTTTATTTTGTTAATGAAATAACTTTTA...
10808,NCU16025,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...
10809,NCU16026,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...
10810,NCU16027,ATGATACAAGTAGCTAAAATAATAGGAACAGGGCTAGCTACCACAG...


In [6]:
# 读取gene_information.csv文件
gene_information_df = pd.read_csv('gene_information.csv')

gene_information_df

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,GO,ONTOLOGY,GO_type,KEGG_definition,Knum,EC_number,Description,Pfam,Cazy
0,NCU10129,NC_026501.1,1151.0,2878.0,-,1728.0,5847003.0,XP_001727976,A7UXG2,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN
1,NCU09901,NC_026501.1,3178.0,6443.0,+,3266.0,3874163.0,XP_958016,Q7S0E6,GO:0003676,nucleic acid binding,MF,"Nucleocytoplasmic transport,mRNA surveillance ...",K14325,NaN,Pfam:RRM_6,RRM_1,GH0: Glycoside Hydrolases (GHs)
2,NCU09903,NC_026501.1,9177.0,13334.0,+,4158.0,3874165.0,XP_958018,Q7S0E4,"GO:0016887,GO:0005524","ATPase activity,ATP binding","MF,MF",NaN,K03235,NaN,Chromatin organization modifier domain(NEW1),"ABC_tran,Chromo",NaN
3,NCU11134,NC_026501.1,15930.0,16647.0,+,718.0,5847002.0,XP_001727977,A7UXG3,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN
4,NCU09904,NC_026501.1,17441.0,19615.0,+,2175.0,3874166.0,XP_958019,Q7S0E3,"GO:0005975,GO:0004553","carbohydrate metabolic process,hydrolase activ...","BP,MF",NaN,NaN,NaN,Glycosyl hydrolases family 16,Glyco_hydro_16,GH16_4: Glycoside Hydrolases (GHs)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10585,NCU16024,NC_026614.1,54186.0,54350.0,+,165.0,23681579.0,YP_009126725,Q08656,"GO:0015986,GO:0015078,GO:0000276","ATP synthesis coupled proton transport,proton ...","BP,MF,CC","Metabolic pathways,Oxidative phosphorylation",NaN,NaN,NaN,NaN,NaN
10586,NCU16025,NC_026614.1,55087.0,57247.0,+,2161.0,23681580.0,YP_009126726,P37212,"GO:0045263,GO:0015986,GO:0015078","proton-transporting ATP synthase complex, coup...","CC,BP,MF","Metabolic pathways,Oxidative phosphorylation",K02126,NaN,Mitochondrial membrane ATP synthase (F(1)F(0) ...,ATP-synt_A,NaN
10587,NCU16026,NC_026614.1,55087.0,56529.0,+,1443.0,23681581.0,YP_009126727,M1RV38,"GO:0045263,GO:0015986,GO:0015078,GO:0004519","proton-transporting ATP synthase complex, coup...","CC,BP,MF,MF",NaN,K02126,NaN,Mitochondrial membrane ATP synthase (F(1)F(0) ...,ATP-synt_A,NaN
10588,NCU16027,NC_026614.1,58608.0,58832.0,+,225.0,23681582.0,YP_009126728,Q12635,"GO:0045263,GO:0015986,GO:0015078,GO:0033177,GO...","proton-transporting ATP synthase complex, coup...","CC,BP,MF,CC,BP,MF","Metabolic pathways,Oxidative phosphorylation","K02128,K03880",1.6.5.3,Belongs to the ATPase C chain family(atp9),ATP-synt_C,NaN


In [7]:
# 将gene_information_df与amino_acid_sequence_df合并
gene_information_df = pd.merge(gene_information_df, amino_acid_sequence_df, on='Protein_id', how='left')

In [8]:
gene_information_df

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,GO,ONTOLOGY,GO_type,KEGG_definition,Knum,EC_number,Description,Pfam,Cazy,Amino_acid_sequence
0,NCU10129,NC_026501.1,1151.0,2878.0,-,1728.0,5847003.0,XP_001727976,A7UXG2,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,MPLFSRHAEPEPAPAPVYQEPAPKRHSLFGGSHHRSPSPGSPTYSS...
1,NCU09901,NC_026501.1,3178.0,6443.0,+,3266.0,3874163.0,XP_958016,Q7S0E6,GO:0003676,nucleic acid binding,MF,"Nucleocytoplasmic transport,mRNA surveillance ...",K14325,NaN,Pfam:RRM_6,RRM_1,GH0: Glycoside Hydrolases (GHs),MASRSRSRSADRYDDDRSRSHTPRRHRSLTPRSARDYSREPSPALG...
2,NCU09903,NC_026501.1,9177.0,13334.0,+,4158.0,3874165.0,XP_958018,Q7S0E4,"GO:0016887,GO:0005524","ATPase activity,ATP binding","MF,MF",NaN,K03235,NaN,Chromatin organization modifier domain(NEW1),"ABC_tran,Chromo",NaN,MTVPIMVADGAAPPVSQADVASILDTVFNAKTSQASIDAAYGLCEV...
3,NCU11134,NC_026501.1,15930.0,16647.0,+,718.0,5847002.0,XP_001727977,A7UXG3,NaN,NaN,NaN,NaN,NaN,NaN,-,NaN,NaN,MVVQAPMLQEPSLVPRVRGNVGMSVKDGPNPGKMMIFNTSRFLVLI...
4,NCU09904,NC_026501.1,17441.0,19615.0,+,2175.0,3874166.0,XP_958019,Q7S0E3,"GO:0005975,GO:0004553","carbohydrate metabolic process,hydrolase activ...","BP,MF",NaN,NaN,NaN,Glycosyl hydrolases family 16,Glyco_hydro_16,GH16_4: Glycoside Hydrolases (GHs),MDHRHTMDNDHREHLDPRASGRHNFDSGVESRASQRSNEGSNENPF...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10585,NCU16024,NC_026614.1,54186.0,54350.0,+,165.0,23681579.0,YP_009126725,Q08656,"GO:0015986,GO:0015078,GO:0000276","ATP synthesis coupled proton transport,proton ...","BP,MF,CC","Metabolic pathways,Oxidative phosphorylation",NaN,NaN,NaN,NaN,NaN,MPQLVPFYFVNEITFTFVIITLMVYILSKYILPRFVRLFLSRTFIS...
10586,NCU16025,NC_026614.1,55087.0,57247.0,+,2161.0,23681580.0,YP_009126726,P37212,"GO:0045263,GO:0015986,GO:0015078","proton-transporting ATP synthase complex, coup...","CC,BP,MF","Metabolic pathways,Oxidative phosphorylation",K02126,NaN,Mitochondrial membrane ATP synthase (F(1)F(0) ...,ATP-synt_A,NaN,MSTLSFNNISTEVLSPLNQFEIRDLLSIDALGNLHISITNIGFYLT...
10587,NCU16026,NC_026614.1,55087.0,56529.0,+,1443.0,23681581.0,YP_009126727,M1RV38,"GO:0045263,GO:0015986,GO:0015078,GO:0004519","proton-transporting ATP synthase complex, coup...","CC,BP,MF,MF",NaN,K02126,NaN,Mitochondrial membrane ATP synthase (F(1)F(0) ...,ATP-synt_A,NaN,MSTLSFNNISTEVLSPLNQFEIRDLLSIDALGNLHISITNIGFYLT...
10588,NCU16027,NC_026614.1,58608.0,58832.0,+,225.0,23681582.0,YP_009126728,Q12635,"GO:0045263,GO:0015986,GO:0015078,GO:0033177,GO...","proton-transporting ATP synthase complex, coup...","CC,BP,MF,CC,BP,MF","Metabolic pathways,Oxidative phosphorylation","K02128,K03880",1.6.5.3,Belongs to the ATPase C chain family(atp9),ATP-synt_C,NaN,MIQVAKIIGTGLATTGLIGAGIGIGVVFGSLIIGVSRNPSLKSQLF...


In [11]:
# 将gene_information_df与nucleic_acid_df合并
gene_information_df = pd.merge(gene_information_df, nucleic_acid_df, on='Gene', how='left')

# 删除Gene列中的重复值
gene_information_df.drop_duplicates(subset=['Gene'], keep='first', inplace=True)

gene_information_df

,Gene,Chr,Start,End,Strand,Length,NCBI_gene_id,Protein_id,Uniport_id,GO,...,KEGG_definition,Knum,EC_number,Description,Pfam,Cazy,Amino_acid_sequence,Nucleic_acid_sequence_x,Nucleic_acid_sequence_y,Nucleic_acid_sequence
0,NCU10129,NC_026501.1,1151.0,2878.0,-,1728.0,5847003.0,XP_001727976,A7UXG2,NaN,...,NaN,NaN,NaN,-,NaN,NaN,MPLFSRHAEPEPAPAPVYQEPAPKRHSLFGGSHHRSPSPGSPTYSS...,ATGCCTCTCTTCTCCCGCCACGCCGAGCCTGAGCCCGCCCCGGCCC...,ATGCCTCTCTTCTCCCGCCACGCCGAGCCTGAGCCCGCCCCGGCCC...,ATGCCTCTCTTCTCCCGCCACGCCGAGCCTGAGCCCGCCCCGGCCC...
1,NCU09901,NC_026501.1,3178.0,6443.0,+,3266.0,3874163.0,XP_958016,Q7S0E6,GO:0003676,...,"Nucleocytoplasmic transport,mRNA surveillance ...",K14325,NaN,Pfam:RRM_6,RRM_1,GH0: Glycoside Hydrolases (GHs),MASRSRSRSADRYDDDRSRSHTPRRHRSLTPRSARDYSREPSPALG...,ATGGCCTCCCGCTCGAGGTCTCGGTCGGCCGACCGCTACGATGACG...,ATGGCCTCCCGCTCGAGGTCTCGGTCGGCCGACCGCTACGATGACG...,ATGGCCTCCCGCTCGAGGTCTCGGTCGGCCGACCGCTACGATGACG...
2,NCU09903,NC_026501.1,9177.0,13334.0,+,4158.0,3874165.0,XP_958018,Q7S0E4,"GO:0016887,GO:0005524",...,NaN,K03235,NaN,Chromatin organization modifier domain(NEW1),"ABC_tran,Chromo",NaN,MTVPIMVADGAAPPVSQADVASILDTVFNAKTSQASIDAAYGLCEV...,ATGACAGTCCCTATCATGGTCGCTGATGGCGCCGCGCCGCCCGTCA...,ATGACAGTCCCTATCATGGTCGCTGATGGCGCCGCGCCGCCCGTCA...,ATGACAGTCCCTATCATGGTCGCTGATGGCGCCGCGCCGCCCGTCA...
3,NCU11134,NC_026501.1,15930.0,16647.0,+,718.0,5847002.0,XP_001727977,A7UXG3,NaN,...,NaN,NaN,NaN,-,NaN,NaN,MVVQAPMLQEPSLVPRVRGNVGMSVKDGPNPGKMMIFNTSRFLVLI...,ATGGTTGTTCAAGCTCCCATGCTACAAGAGCCCTCACTGGTTCCAC...,ATGGTTGTTCAAGCTCCCATGCTACAAGAGCCCTCACTGGTTCCAC...,ATGGTTGTTCAAGCTCCCATGCTACAAGAGCCCTCACTGGTTCCAC...
4,NCU09904,NC_026501.1,17441.0,19615.0,+,2175.0,3874166.0,XP_958019,Q7S0E3,"GO:0005975,GO:0004553",...,NaN,NaN,NaN,Glycosyl hydrolases family 16,Glyco_hydro_16,GH16_4: Glycoside Hydrolases (GHs),MDHRHTMDNDHREHLDPRASGRHNFDSGVESRASQRSNEGSNENPF...,ATGGACCACAGACACACCATGGACAACGACCACCGCGAACATCTCG...,ATGGACCACAGACACACCATGGACAACGACCACCGCGAACATCTCG...,ATGGACCACAGACACACCATGGACAACGACCACCGCGAACATCTCG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20394,NCU16024,NC_026614.1,54186.0,54350.0,+,165.0,23681579.0,YP_009126725,Q08656,"GO:0015986,GO:0015078,GO:0000276",...,"Metabolic pathways,Oxidative phosphorylation",NaN,NaN,NaN,NaN,NaN,MPQLVPFYFVNEITFTFVIITLMVYILSKYILPRFVRLFLSRTFIS...,ATGCCTCAATTAGTTCCATTTTATTTTGTTAATGAAATAACTTTTA...,ATGCCTCAATTAGTTCCATTTTATTTTGTTAATGAAATAACTTTTA...,ATGCCTCAATTAGTTCCATTTTATTTTGTTAATGAAATAACTTTTA...
20395,NCU16025,NC_026614.1,55087.0,57247.0,+,2161.0,23681580.0,YP_009126726,P37212,"GO:0045263,GO:0015986,GO:0015078",...,"Metabolic pathways,Oxidative phosphorylation",K02126,NaN,Mitochondrial membrane ATP synthase (F(1)F(0) ...,ATP-synt_A,NaN,MSTLSFNNISTEVLSPLNQFEIRDLLSIDALGNLHISITNIGFYLT...,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...
20396,NCU16026,NC_026614.1,55087.0,56529.0,+,1443.0,23681581.0,YP_009126727,M1RV38,"GO:0045263,GO:0015986,GO:0015078,GO:0004519",...,NaN,K02126,NaN,Mitochondrial membrane ATP synthase (F(1)F(0) ...,ATP-synt_A,NaN,MSTLSFNNISTEVLSPLNQFEIRDLLSIDALGNLHISITNIGFYLT...,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...,ATGAGTACTTTAAGCTTTAATAATATATCGACAGAAGTCCTTAGCC...
20397,NCU16027,NC_026614.1,58608.0,58832.0,+,225.0,23681582.0,YP_009126728,Q12635,"GO:0045263,GO:0015986,GO:0015078,GO:0033177,GO...",...,"Metabolic pathways,Oxidative phosphorylation","K02128,K03880",1.6.5.3,Belongs to the ATPase C chain family(atp9),ATP-synt_C,NaN,MIQVAKIIGTGLATTGLIGAGIGIGVVFGSLIIGVSRNPSLKSQLF...,ATGATACAAGTAGCTAAAATAATAGGAACAGGGCTAGCTACCACAG...,ATGATACAAGTAGCTAAAATAATAGGAACAGGGCTAGCTACCACAG...,ATGATACAAGTAGCTAAAATAATAGGAACAGGGCTAGCTACCACAG...
